# SA-LIB AI Workshop 2026 — Day 1: from simple RAG to agentic RAG (standalone code)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dhelamay/CPUT_ICAILY_AI2026_MaganLLM_WORKSHOP/blob/main/SA_LIB_WORKSHOP_Day1/notebooks/SA_LIB_Day1_RAG_Standalone.ipynb)

**Author:** Dr. Nasser Mooman · Magan AI Inc. · nmooman@gmail.com · written with help from Claude (Anthropic)

> ⚠️ **Teaching code — do not use in production.** Short on purpose: no authentication, input validation, prompt-injection protection, error handling or tests. LLM answers can be wrong. Provided as is, without warranty.

**Standalone version:** the same 12 examples, but every code cell is **complete**. It loads the data, builds the index, calls the LLM and runs the technique, **without** `rag_common.py`. You can run any example on its own after the setup cells. The setup part repeats in every cell on purpose: read it once in **01**, then jump to *the technique* in the others. Embeddings are always the free local model.

| # | technique | idea in one line |
|---|---|---|
| 00 | no RAG | the problem: the LLM doesn't know your documents |
| 01 | naive RAG | index → retrieve → generate |
| 02 | hybrid search | keywords (BM25) + meaning (vectors), fused with RRF |
| 03 | re-ranking | a cross-encoder re-scores the top candidates |
| 04 | query transformation | multi-query + HyDE rewrite the question before searching |
| 05 | conversational RAG | memory + standalone-question rewriting |
| 06 | corrective RAG (CRAG) | grade retrieved chunks, fall back to web search |
| 07 | self-RAG | the model checks its own answer and retries |
| 08 | graph RAG | LLM-built knowledge graph for multi-hop questions |
| 09 | adaptive RAG | a router picks: no retrieval / docs / web |
| 10 | agentic RAG | the LLM drives a tool loop (no framework) |
| 11 | agentic RAG + LangGraph | the same agent as a graph, with memory |

## 0. Setup

In [ ]:
%pip install -q "openai>=1.40" "python-dotenv" "numpy" "fastembed>=0.4" "pypdf" "rank-bm25" "networkx" "ddgs" "langgraph>=1.0" "langchain-openai>=1.0"

### API keys and provider
**Bring your own LLM API key** (Groq and Gemini have free tiers). No key is needed only if you already run a local server (Ollama, SGLang or vLLM): set `LLM_PROVIDER=ollama`, `OLLAMA_BASE_URL` and `LLM_MODEL`.

* **Colab:** 🔑 *Secrets* panel → add e.g. `GROQ_API_KEY` or `DEEPSEEK_API_KEY` or `OPENAI_API_KEY`.
* **VS Code / Linux:** put keys in `.env` (copy `.env.example`).

`LLM_PROVIDER`: `openai` · `deepseek` · `groq` · `gemini` · `openrouter` · `ollama`  
`EMBEDDING_PROVIDER`: `local` (free, on CPU) · `openai` · `gemini` · `ollama`

In [ ]:
import os, getpass
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
try:
    from google.colab import userdata
    for key in ['GROQ_API_KEY', 'GOOGLE_API_KEY', 'OPENAI_API_KEY', 'DEEPSEEK_API_KEY', 'OPENROUTER_API_KEY', 'LLM_PROVIDER', 'LLM_MODEL', 'EMBEDDING_PROVIDER']:
        try:
            os.environ[key] = userdata.get(key)
        except Exception:
            pass
except ImportError:
    pass

os.environ.setdefault("LLM_PROVIDER", "groq")        # <- or "deepseek", "openai", "gemini" ...
os.environ.setdefault("EMBEDDING_PROVIDER", "local")  # free local embeddings
provider = os.environ["LLM_PROVIDER"]
key_name = {"groq": "GROQ_API_KEY", "gemini": "GOOGLE_API_KEY", "openai": "OPENAI_API_KEY",
            "deepseek": "DEEPSEEK_API_KEY", "openrouter": "OPENROUTER_API_KEY"}.get(provider)
if key_name and not os.getenv(key_name):
    os.environ[key_name] = getpass.getpass(f"{key_name}: ")
print("llm:", provider, "| embeddings:", os.environ["EMBEDDING_PROVIDER"])

### The sample documents (`data/`)
A fictional solar company, *Sahara Sun Energy*, so the LLM **cannot** know the answers without RAG. Replace or add your own `.md`, `.txt` or `.pdf` files in `data/`.

In [ ]:
import os
os.makedirs("data", exist_ok=True)

In [ ]:
%%writefile data/01_company_overview.md
# Sahara Sun Energy — Company Overview

Sahara Sun Energy (SSE) is a fictional solar-power company created for the SA-LIB AI Workshop.
It was founded in 2019 in Benghazi, Libya, by Dr. Amira Belhaj (CEO) and Omar Zway (CTO).

## Mission
SSE's mission is to bring reliable, affordable solar electricity to towns and farms across
eastern and southern Libya, and to train local engineers to maintain it.

## Headquarters and offices
- Headquarters: Benghazi (Al-Hadaek district).
- Regional office: Sabha, opened in 2022, led by regional manager Khaled Senussi.
- Research lab: Tripoli, opened in 2024, led by Dr. Laila Mansour.

## Size
As of January 2026 SSE employs 214 people: 131 in Benghazi, 52 in Sabha and 31 in Tripoli.

## Flagship projects
- **Kufra Solar Farm** — a 40 MW solar farm near Kufra, completed in March 2023.
  Project lead: Omar Zway.
- **Green Wells Program** — solar-powered water pumps for 120 farms around Sabha.
  Project lead: Khaled Senussi. Started 2022; 87 farms connected by end of 2025.
- **SunGrid Pilot** — a micro-grid with battery storage for the old town of Ghadames.
  Project lead: Dr. Laila Mansour. Planned completion: Q3 2026.

## Partners
SSE partners with the University of Benghazi (engineering internships) and with the fictional
Mediterranean Clean Energy Fund (MCEF), which financed 60% of the Kufra Solar Farm.

In [ ]:
%%writefile data/02_products.md
# Sahara Sun Energy — Products and Prices

All prices are in Libyan dinar (LYD) and were valid on 1 January 2026.

## SunBox Home (SB-H1)
A rooftop kit for a family home.
- Panels: 6 x 450 W (2.7 kW total).
- Battery: 5 kWh lithium iron phosphate (LiFePO4).
- Price: 18,500 LYD installed.
- Warranty: 10 years on panels, 5 years on battery, 2 years on the inverter.

## SunBox Farm (SB-F3)
A kit designed for farms that need to pump water.
- Panels: 12 x 450 W (5.4 kW total).
- Includes a 3 HP solar water pump controller.
- Price: 34,000 LYD installed.
- Warranty: 10 years on panels, 3 years on the pump controller.

## SunGrid Community (SG-C)
A micro-grid for a village or neighbourhood of 50–300 homes. Price is quoted per project.
Typical projects cost between 1.2 and 4.5 million LYD.

## Maintenance plan
The optional "SunCare" plan costs 900 LYD per year for SunBox Home and 1,600 LYD per year for
SunBox Farm. It includes two cleaning and inspection visits per year and priority repairs
within 72 hours.

## Payment
Customers can pay in full or use the "Pay-as-you-save" plan: 20% deposit and the rest in
24 monthly instalments with no interest.

In [ ]:
%%writefile data/03_hr_policy.md
# Sahara Sun Energy — Employee Handbook (HR Policy)

## Working hours
The standard working week is Sunday to Thursday, 8:00 to 16:00. Field teams working at solar
farms may use a 4-day shift pattern approved by their regional manager.

## Annual leave
- Employees receive 30 days of paid annual leave per year.
- Employees with more than 5 years of service receive 35 days.
- Up to 10 unused days can be carried over to the next year; the rest expires on 31 March.
- Leave requests must be submitted in the HR portal at least 14 days in advance.

## Sick leave
Up to 15 days of paid sick leave per year. A medical certificate is required for absences
longer than 2 consecutive days.

## Remote work
Office staff may work remotely up to 2 days per week with manager approval. Field engineers
are not eligible for remote work.

## Training budget
Every employee has a training budget of 3,000 LYD per year for courses and certifications.
AI and data-science courses (like the SA-LIB workshop) are pre-approved.

## Safety
Anyone working on rooftops or solar farms must complete the "SunSafe" safety course before
their first field visit and renew it every 2 years.

In [ ]:
%%writefile data/04_research_lab.md
# Sahara Sun Energy — Tripoli Research Lab

The Tripoli Research Lab opened in 2024 and is led by Dr. Laila Mansour, a battery chemist who
previously worked at a university lab in Tunis.

## Research topics
1. **Dust mitigation**: Saharan dust can reduce panel output by up to 25% in a single week.
   The lab tests hydrophobic coatings and robotic dry-cleaning brushes. Their "DuneGuard"
   coating reduced losses to about 7% in 2025 field tests at Kufra.
2. **Heat-tolerant batteries**: summer temperatures above 45 °C shorten battery life. The lab
   works on passive cooling enclosures for LiFePO4 batteries.
3. **AI forecasting**: a machine-learning model that predicts next-day solar output from
   weather data. Its average forecast error in 2025 was 6.2%. The model is used to schedule
   battery charging on the SunGrid Pilot in Ghadames.

## Team
- Dr. Laila Mansour — lab director.
- Youssef Ettir — data scientist, built the AI forecasting model.
- Salma Gheriani — materials engineer, leads the DuneGuard coating work.

## Collaboration
The lab collaborates with the University of Benghazi and hosts 6 master's students per year.

In [ ]:
%%writefile data/05_workshop_handbook.md
# SA-LIB AI Workshop 2026 — Participant Handbook

## Schedule
- **Day 1 — Retrieval-Augmented Generation (RAG):** from naive RAG to agentic RAG.
- **Day 2 — AI Agents:** the same agent built in LangChain, LangGraph, CrewAI, OpenAI Agents SDK,
  Google ADK, Microsoft Agent Framework and MLflow, plus multi-agent A2A.

Sessions run from 9:00 to 16:30 with a lunch break from 12:30 to 13:30.

## What to bring
- A laptop with Python 3.10 or newer (or a Google account to use Google Colab).
- At least one LLM API key. Free options: Groq and Google Gemini. Low-cost options: DeepSeek.
  OpenAI is also supported.

## Rules for API keys
Never paste API keys into notebooks you share, and never commit the `.env` file to GitHub.

## Certificates
Participants who complete the Day 1 and Day 2 exercises receive a certificate of completion.

In [ ]:
%%writefile data/06_faq.md
# Sahara Sun Energy — Customer FAQ

**How long does installation take?**
A SunBox Home is usually installed in one day. A SunBox Farm takes two days.

**Do panels work during sandstorms?**
Output drops during a sandstorm and while panels are dusty. We recommend cleaning every 2–4
weeks in the dry season, or choosing the SunCare maintenance plan.

**What happens at night?**
The battery supplies power at night. A 5 kWh battery typically powers lights, a fridge, fans
and a TV for 8–10 hours.

**Which cities do you serve?**
Benghazi, Tripoli, Sabha, Kufra, Ghadames and surrounding areas. Other locations on request.

**How do I contact support?**
Call the support line 1515 (Sunday–Thursday, 8:00–20:00) or email support@saharasun.example.

## 00. `00_no_rag_baseline.py`

00 — the problem RAG solves
ask the LLM about our (fictional) company WITHOUT giving it any documents.
it has never seen these files, so it will guess, refuse, or hallucinate.

In [ ]:
import os

from dotenv import find_dotenv, load_dotenv
from openai import OpenAI

load_dotenv(find_dotenv(usecwd=True))   # read LLM_PROVIDER and your API key from the .env file

# ============ setup: the LLM (every provider speaks the OpenAI protocol, so one client works for all) ============
PROVIDERS = {  # name: (base_url, api-key env var, default model)
    "openai":     ("https://api.openai.com/v1", "OPENAI_API_KEY", "gpt-4o-mini"),
    "deepseek":   ("https://api.deepseek.com", "DEEPSEEK_API_KEY", "deepseek-chat"),
    "groq":       ("https://api.groq.com/openai/v1", "GROQ_API_KEY", "llama-3.3-70b-versatile"),
    "gemini":     ("https://generativelanguage.googleapis.com/v1beta/openai/", "GOOGLE_API_KEY", "gemini-2.5-flash"),
    "openrouter": ("https://openrouter.ai/api/v1", "OPENROUTER_API_KEY", "meta-llama/llama-3.3-70b-instruct:free"),
    "ollama":     (os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1"), None, "qwen2.5:7b"),
}
PROVIDER = os.getenv("LLM_PROVIDER", "groq").strip().lower()
BASE_URL, KEY_ENV, DEFAULT_MODEL = PROVIDERS[PROVIDER]
if KEY_ENV and not os.getenv(KEY_ENV):
    raise SystemExit(f"{KEY_ENV} is missing. Add it to your .env file (see .env.example).")
client = OpenAI(base_url=BASE_URL, api_key=os.getenv(KEY_ENV) if KEY_ENV else "ollama")
MODEL = os.getenv("LLM_MODEL") or DEFAULT_MODEL


def chat(prompt: str, system: str = "You are a helpful assistant.") -> str:
    """Send one prompt to the LLM and return its text answer."""
    resp = client.chat.completions.create(model=MODEL, temperature=0, messages=[
        {"role": "system", "content": system}, {"role": "user", "content": prompt}])
    return resp.choices[0].message.content.strip()


def show(title: str, text: str) -> None:
    print(f"\n{'=' * 70}\n{title}\n{'-' * 70}\n{text}")


# ============ the example ============
QUESTION = "How many days of annual leave does a Sahara Sun Energy employee with 7 years of service get?"

show("question", QUESTION)
show("answer WITHOUT retrieval (the model is guessing)", chat(QUESTION))

## 01. `01_naive_rag.py`

01 — naive (simple) RAG
the classic 3 steps:  INDEX  ->  RETRIEVE  ->  GENERATE
  index:    load files, cut them into chunks, turn each chunk into a vector
  retrieve: embed the question, find the chunks with the closest vectors
  generate: give those chunks to the LLM as context and ask it to answer

In [ ]:
import os
from pathlib import Path

import numpy as np
from dotenv import find_dotenv, load_dotenv
from fastembed import TextEmbedding
from openai import OpenAI

load_dotenv(find_dotenv(usecwd=True))   # read LLM_PROVIDER and your API key from the .env file

# ============ setup: the LLM (every provider speaks the OpenAI protocol, so one client works for all) ============
PROVIDERS = {  # name: (base_url, api-key env var, default model)
    "openai":     ("https://api.openai.com/v1", "OPENAI_API_KEY", "gpt-4o-mini"),
    "deepseek":   ("https://api.deepseek.com", "DEEPSEEK_API_KEY", "deepseek-chat"),
    "groq":       ("https://api.groq.com/openai/v1", "GROQ_API_KEY", "llama-3.3-70b-versatile"),
    "gemini":     ("https://generativelanguage.googleapis.com/v1beta/openai/", "GOOGLE_API_KEY", "gemini-2.5-flash"),
    "openrouter": ("https://openrouter.ai/api/v1", "OPENROUTER_API_KEY", "meta-llama/llama-3.3-70b-instruct:free"),
    "ollama":     (os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1"), None, "qwen2.5:7b"),
}
PROVIDER = os.getenv("LLM_PROVIDER", "groq").strip().lower()
BASE_URL, KEY_ENV, DEFAULT_MODEL = PROVIDERS[PROVIDER]
if KEY_ENV and not os.getenv(KEY_ENV):
    raise SystemExit(f"{KEY_ENV} is missing. Add it to your .env file (see .env.example).")
client = OpenAI(base_url=BASE_URL, api_key=os.getenv(KEY_ENV) if KEY_ENV else "ollama")
MODEL = os.getenv("LLM_MODEL") or DEFAULT_MODEL


def chat(prompt: str, system: str = "You are a helpful assistant.") -> str:
    """Send one prompt to the LLM and return its text answer."""
    resp = client.chat.completions.create(model=MODEL, temperature=0, messages=[
        {"role": "system", "content": system}, {"role": "user", "content": prompt}])
    return resp.choices[0].message.content.strip()


def show(title: str, text: str) -> None:
    print(f"\n{'=' * 70}\n{title}\n{'-' * 70}\n{text}")


# ============ step 1 — INDEX: load data/ -> cut into chunks -> turn every chunk into a vector ============
DATA_DIR = Path("data")   # written by the data/ cells at the top of this notebook


def chunk_text(text: str, chunk_size: int = 120, overlap: int = 20) -> list[str]:
    """Split text into chunks of ~chunk_size words; neighbouring chunks share `overlap` words."""
    words = text.split()
    if len(words) <= chunk_size:
        return [text.strip()]
    return [" ".join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size - overlap)]


def load_chunks(chunk_size: int = 120, overlap: int = 20) -> list[dict]:
    """Read every .md / .txt / .pdf file in data/ and chunk it. Each chunk remembers its file."""
    chunks = []
    for path in sorted(DATA_DIR.iterdir()):
        if path.suffix.lower() in {".md", ".txt"}:
            text = path.read_text(encoding="utf-8")
        elif path.suffix.lower() == ".pdf":
            from pypdf import PdfReader   # only needed when you put PDF files in data/
            text = "\n".join(page.extract_text() or "" for page in PdfReader(path).pages)
        else:
            continue
        for i, piece in enumerate(chunk_text(text, chunk_size, overlap)):
            chunks.append({"id": f"{path.name}#{i}", "source": path.name, "text": piece})
    return chunks


embedder = TextEmbedding("BAAI/bge-small-en-v1.5")   # free, runs on your CPU (small download the first time)


def embed(texts: list[str]) -> np.ndarray:
    """Texts -> a (n, dim) array of vectors, each scaled to length 1."""
    vectors = np.array(list(embedder.embed(texts)))
    return vectors / np.linalg.norm(vectors, axis=1, keepdims=True)


chunks = load_chunks(chunk_size=120, overlap=20)
vectors = embed([c["text"] for c in chunks])          # our whole "vector database": one numpy array
print(f"[index] {len(chunks)} chunks from {len({c['source'] for c in chunks})} files "
      f"| embeddings: local | llm: {PROVIDER}")


# ============ step 2 — RETRIEVE: the chunks whose vectors are closest to the question's vector ============
def search(query: str, k: int = 4) -> list[dict]:
    """Return the k most similar chunks, each with a 'score' (cosine similarity)."""
    scores = vectors @ embed([query])[0]              # dot product of length-1 vectors = cosine similarity
    return [{**chunks[i], "score": float(scores[i])} for i in np.argsort(-scores)[:k]]


# ============ step 3 — GENERATE: the LLM answers using ONLY the retrieved chunks ============
ANSWER_SYSTEM = ("You answer questions using ONLY the provided context. "
                 "Cite sources like [1], [2]. If the answer is not in the context, say \"I don't know\".")


def answer_from_context(question: str, hits: list[dict]) -> str:
    context = "\n\n".join(f"[{i + 1}] (source: {h['source']})\n{h['text']}" for i, h in enumerate(hits))
    return chat(f"Context:\n{context}\n\nQuestion: {question}", system=ANSWER_SYSTEM)


# ============ the example ============
QUESTION = "How many days of annual leave does a Sahara Sun Energy employee with 7 years of service get?"

hits = search(QUESTION, k=3)
show("retrieved chunks", "\n".join(f"{h['score']:.3f}  {h['id']}" for h in hits))
show("answer WITH retrieval", answer_from_context(QUESTION, hits))

## 02. `02_hybrid_search_rag.py`

02 — hybrid search RAG  (keywords + meaning)
vector search understands MEANING ("holiday" ~ "annual leave") but can miss exact tokens
like product codes ("SB-F3") or numbers. BM25 keyword search is the opposite.
hybrid = run both, then merge the two rankings with Reciprocal Rank Fusion (RRF).

In [ ]:
import os
from pathlib import Path

import numpy as np
from dotenv import find_dotenv, load_dotenv
from fastembed import TextEmbedding
from openai import OpenAI
from rank_bm25 import BM25Okapi

load_dotenv(find_dotenv(usecwd=True))   # read LLM_PROVIDER and your API key from the .env file

# ============ setup: the LLM (every provider speaks the OpenAI protocol, so one client works for all) ============
PROVIDERS = {  # name: (base_url, api-key env var, default model)
    "openai":     ("https://api.openai.com/v1", "OPENAI_API_KEY", "gpt-4o-mini"),
    "deepseek":   ("https://api.deepseek.com", "DEEPSEEK_API_KEY", "deepseek-chat"),
    "groq":       ("https://api.groq.com/openai/v1", "GROQ_API_KEY", "llama-3.3-70b-versatile"),
    "gemini":     ("https://generativelanguage.googleapis.com/v1beta/openai/", "GOOGLE_API_KEY", "gemini-2.5-flash"),
    "openrouter": ("https://openrouter.ai/api/v1", "OPENROUTER_API_KEY", "meta-llama/llama-3.3-70b-instruct:free"),
    "ollama":     (os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1"), None, "qwen2.5:7b"),
}
PROVIDER = os.getenv("LLM_PROVIDER", "groq").strip().lower()
BASE_URL, KEY_ENV, DEFAULT_MODEL = PROVIDERS[PROVIDER]
if KEY_ENV and not os.getenv(KEY_ENV):
    raise SystemExit(f"{KEY_ENV} is missing. Add it to your .env file (see .env.example).")
client = OpenAI(base_url=BASE_URL, api_key=os.getenv(KEY_ENV) if KEY_ENV else "ollama")
MODEL = os.getenv("LLM_MODEL") or DEFAULT_MODEL


def chat(prompt: str, system: str = "You are a helpful assistant.") -> str:
    """Send one prompt to the LLM and return its text answer."""
    resp = client.chat.completions.create(model=MODEL, temperature=0, messages=[
        {"role": "system", "content": system}, {"role": "user", "content": prompt}])
    return resp.choices[0].message.content.strip()


def show(title: str, text: str) -> None:
    print(f"\n{'=' * 70}\n{title}\n{'-' * 70}\n{text}")


# ============ setup: INDEX = load data/ -> cut into chunks -> turn every chunk into a vector ============
DATA_DIR = Path("data")   # written by the data/ cells at the top of this notebook


def chunk_text(text: str, chunk_size: int = 120, overlap: int = 20) -> list[str]:
    """Split text into chunks of ~chunk_size words; neighbouring chunks share `overlap` words."""
    words = text.split()
    if len(words) <= chunk_size:
        return [text.strip()]
    return [" ".join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size - overlap)]


def load_chunks(chunk_size: int = 120, overlap: int = 20) -> list[dict]:
    """Read every .md / .txt / .pdf file in data/ and chunk it. Each chunk remembers its file."""
    chunks = []
    for path in sorted(DATA_DIR.iterdir()):
        if path.suffix.lower() in {".md", ".txt"}:
            text = path.read_text(encoding="utf-8")
        elif path.suffix.lower() == ".pdf":
            from pypdf import PdfReader   # only needed when you put PDF files in data/
            text = "\n".join(page.extract_text() or "" for page in PdfReader(path).pages)
        else:
            continue
        for i, piece in enumerate(chunk_text(text, chunk_size, overlap)):
            chunks.append({"id": f"{path.name}#{i}", "source": path.name, "text": piece})
    return chunks


embedder = TextEmbedding("BAAI/bge-small-en-v1.5")   # free, runs on your CPU (small download the first time)


def embed(texts: list[str]) -> np.ndarray:
    """Texts -> a (n, dim) array of vectors, each scaled to length 1."""
    vectors = np.array(list(embedder.embed(texts)))
    return vectors / np.linalg.norm(vectors, axis=1, keepdims=True)


chunks = load_chunks()
vectors = embed([c["text"] for c in chunks])          # our whole "vector database": one numpy array
print(f"[index] {len(chunks)} chunks from {len({c['source'] for c in chunks})} files "
      f"| embeddings: local | llm: {PROVIDER}")


def search(query: str, k: int = 4) -> list[dict]:
    """VECTOR search: the k chunks whose meaning is closest to the query."""
    scores = vectors @ embed([query])[0]              # dot product of length-1 vectors = cosine similarity
    return [{**chunks[i], "score": float(scores[i])} for i in np.argsort(-scores)[:k]]


ANSWER_SYSTEM = ("You answer questions using ONLY the provided context. "
                 "Cite sources like [1], [2]. If the answer is not in the context, say \"I don't know\".")


def answer_from_context(question: str, hits: list[dict]) -> str:
    context = "\n\n".join(f"[{i + 1}] (source: {h['source']})\n{h['text']}" for i, h in enumerate(hits))
    return chat(f"Context:\n{context}\n\nQuestion: {question}", system=ANSWER_SYSTEM)


# ============ the technique: KEYWORD search (BM25) + fusion ============
bm25 = BM25Okapi([c["text"].lower().split() for c in chunks])


def keyword_search(query: str, k: int = 4) -> list[dict]:
    scores = bm25.get_scores(query.lower().split())
    best = sorted(range(len(scores)), key=lambda i: -scores[i])[:k]
    return [chunks[i] for i in best]


def hybrid_search(query: str, k: int = 4, rrf_k: int = 60) -> list[dict]:
    """Reciprocal Rank Fusion: score = sum over rankings of 1 / (rrf_k + rank)."""
    fused: dict[str, float] = {}
    by_id = {}
    for ranking in (search(query, k=10), keyword_search(query, k=10)):
        for rank, chunk in enumerate(ranking):
            fused[chunk["id"]] = fused.get(chunk["id"], 0) + 1 / (rrf_k + rank)
            by_id[chunk["id"]] = chunk
    best = sorted(fused, key=lambda cid: -fused[cid])[:k]
    return [by_id[cid] for cid in best]


# ============ the example ============
QUESTION = "What is the warranty on the SB-F3 pump controller and how much does SB-F3 cost?"

show("vector only", "\n".join(c["id"] for c in search(QUESTION, k=4)))
show("keyword (BM25) only", "\n".join(c["id"] for c in keyword_search(QUESTION)))
hits = hybrid_search(QUESTION)
show("hybrid (RRF fused)", "\n".join(c["id"] for c in hits))
show("answer", answer_from_context(QUESTION, hits))

## 03. `03_reranking_rag.py`

03 — RAG with re-ranking
step 1: retrieve MANY candidates quickly (vector search, k=10)
step 2: re-score each (question, chunk) pair with a slower but smarter CROSS-ENCODER
step 3: keep only the top 3 for the LLM
the cross-encoder reads the question and chunk TOGETHER, so it judges relevance much better.
runs locally and free (fastembed, ~80 MB model).

In [ ]:
import os
from pathlib import Path

import numpy as np
from dotenv import find_dotenv, load_dotenv
from fastembed import TextEmbedding
from fastembed.rerank.cross_encoder import TextCrossEncoder
from openai import OpenAI

load_dotenv(find_dotenv(usecwd=True))   # read LLM_PROVIDER and your API key from the .env file

# ============ setup: the LLM (every provider speaks the OpenAI protocol, so one client works for all) ============
PROVIDERS = {  # name: (base_url, api-key env var, default model)
    "openai":     ("https://api.openai.com/v1", "OPENAI_API_KEY", "gpt-4o-mini"),
    "deepseek":   ("https://api.deepseek.com", "DEEPSEEK_API_KEY", "deepseek-chat"),
    "groq":       ("https://api.groq.com/openai/v1", "GROQ_API_KEY", "llama-3.3-70b-versatile"),
    "gemini":     ("https://generativelanguage.googleapis.com/v1beta/openai/", "GOOGLE_API_KEY", "gemini-2.5-flash"),
    "openrouter": ("https://openrouter.ai/api/v1", "OPENROUTER_API_KEY", "meta-llama/llama-3.3-70b-instruct:free"),
    "ollama":     (os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1"), None, "qwen2.5:7b"),
}
PROVIDER = os.getenv("LLM_PROVIDER", "groq").strip().lower()
BASE_URL, KEY_ENV, DEFAULT_MODEL = PROVIDERS[PROVIDER]
if KEY_ENV and not os.getenv(KEY_ENV):
    raise SystemExit(f"{KEY_ENV} is missing. Add it to your .env file (see .env.example).")
client = OpenAI(base_url=BASE_URL, api_key=os.getenv(KEY_ENV) if KEY_ENV else "ollama")
MODEL = os.getenv("LLM_MODEL") or DEFAULT_MODEL


def chat(prompt: str, system: str = "You are a helpful assistant.") -> str:
    """Send one prompt to the LLM and return its text answer."""
    resp = client.chat.completions.create(model=MODEL, temperature=0, messages=[
        {"role": "system", "content": system}, {"role": "user", "content": prompt}])
    return resp.choices[0].message.content.strip()


def show(title: str, text: str) -> None:
    print(f"\n{'=' * 70}\n{title}\n{'-' * 70}\n{text}")


# ============ setup: INDEX = load data/ -> cut into chunks -> turn every chunk into a vector ============
DATA_DIR = Path("data")   # written by the data/ cells at the top of this notebook


def chunk_text(text: str, chunk_size: int = 120, overlap: int = 20) -> list[str]:
    """Split text into chunks of ~chunk_size words; neighbouring chunks share `overlap` words."""
    words = text.split()
    if len(words) <= chunk_size:
        return [text.strip()]
    return [" ".join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size - overlap)]


def load_chunks(chunk_size: int = 120, overlap: int = 20) -> list[dict]:
    """Read every .md / .txt / .pdf file in data/ and chunk it. Each chunk remembers its file."""
    chunks = []
    for path in sorted(DATA_DIR.iterdir()):
        if path.suffix.lower() in {".md", ".txt"}:
            text = path.read_text(encoding="utf-8")
        elif path.suffix.lower() == ".pdf":
            from pypdf import PdfReader   # only needed when you put PDF files in data/
            text = "\n".join(page.extract_text() or "" for page in PdfReader(path).pages)
        else:
            continue
        for i, piece in enumerate(chunk_text(text, chunk_size, overlap)):
            chunks.append({"id": f"{path.name}#{i}", "source": path.name, "text": piece})
    return chunks


embedder = TextEmbedding("BAAI/bge-small-en-v1.5")   # free, runs on your CPU (small download the first time)


def embed(texts: list[str]) -> np.ndarray:
    """Texts -> a (n, dim) array of vectors, each scaled to length 1."""
    vectors = np.array(list(embedder.embed(texts)))
    return vectors / np.linalg.norm(vectors, axis=1, keepdims=True)


chunks = load_chunks()
vectors = embed([c["text"] for c in chunks])          # our whole "vector database": one numpy array
print(f"[index] {len(chunks)} chunks from {len({c['source'] for c in chunks})} files "
      f"| embeddings: local | llm: {PROVIDER}")


def search(query: str, k: int = 4) -> list[dict]:
    """Return the k most similar chunks, each with a 'score' (cosine similarity)."""
    scores = vectors @ embed([query])[0]              # dot product of length-1 vectors = cosine similarity
    return [{**chunks[i], "score": float(scores[i])} for i in np.argsort(-scores)[:k]]


ANSWER_SYSTEM = ("You answer questions using ONLY the provided context. "
                 "Cite sources like [1], [2]. If the answer is not in the context, say \"I don't know\".")


def answer_from_context(question: str, hits: list[dict]) -> str:
    context = "\n\n".join(f"[{i + 1}] (source: {h['source']})\n{h['text']}" for i, h in enumerate(hits))
    return chat(f"Context:\n{context}\n\nQuestion: {question}", system=ANSWER_SYSTEM)


# ============ the technique: re-rank the candidates with a cross-encoder ============
reranker = TextCrossEncoder(model_name="Xenova/ms-marco-MiniLM-L-6-v2")

QUESTION = "Who built the model that predicts solar output, and how accurate is it?"

candidates = search(QUESTION, k=10)
scores = list(reranker.rerank(QUESTION, [c["text"] for c in candidates]))
reranked = [c for _, c in sorted(zip(scores, candidates), key=lambda p: -p[0])][:3]

show("before re-ranking (vector order, top 3)", "\n".join(c["id"] for c in candidates[:3]))
show("after re-ranking (cross-encoder, top 3)", "\n".join(c["id"] for c in reranked))
show("answer", answer_from_context(QUESTION, reranked))

## 04. `04_query_transformation_rag.py`

04 — query transformation RAG  (multi-query + HyDE)
users ask short, vague questions. we let the LLM improve the QUERY before searching:
  multi-query: rewrite the question 3 different ways, search with each, merge results
  HyDE:        let the LLM write a *hypothetical* answer, and search with THAT text
               (a fake answer often looks more like the real document than the question does)

In [ ]:
import os
from pathlib import Path

import numpy as np
from dotenv import find_dotenv, load_dotenv
from fastembed import TextEmbedding
from openai import OpenAI

load_dotenv(find_dotenv(usecwd=True))   # read LLM_PROVIDER and your API key from the .env file

# ============ setup: the LLM (every provider speaks the OpenAI protocol, so one client works for all) ============
PROVIDERS = {  # name: (base_url, api-key env var, default model)
    "openai":     ("https://api.openai.com/v1", "OPENAI_API_KEY", "gpt-4o-mini"),
    "deepseek":   ("https://api.deepseek.com", "DEEPSEEK_API_KEY", "deepseek-chat"),
    "groq":       ("https://api.groq.com/openai/v1", "GROQ_API_KEY", "llama-3.3-70b-versatile"),
    "gemini":     ("https://generativelanguage.googleapis.com/v1beta/openai/", "GOOGLE_API_KEY", "gemini-2.5-flash"),
    "openrouter": ("https://openrouter.ai/api/v1", "OPENROUTER_API_KEY", "meta-llama/llama-3.3-70b-instruct:free"),
    "ollama":     (os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1"), None, "qwen2.5:7b"),
}
PROVIDER = os.getenv("LLM_PROVIDER", "groq").strip().lower()
BASE_URL, KEY_ENV, DEFAULT_MODEL = PROVIDERS[PROVIDER]
if KEY_ENV and not os.getenv(KEY_ENV):
    raise SystemExit(f"{KEY_ENV} is missing. Add it to your .env file (see .env.example).")
client = OpenAI(base_url=BASE_URL, api_key=os.getenv(KEY_ENV) if KEY_ENV else "ollama")
MODEL = os.getenv("LLM_MODEL") or DEFAULT_MODEL


def chat(prompt: str, system: str = "You are a helpful assistant.") -> str:
    """Send one prompt to the LLM and return its text answer."""
    resp = client.chat.completions.create(model=MODEL, temperature=0, messages=[
        {"role": "system", "content": system}, {"role": "user", "content": prompt}])
    return resp.choices[0].message.content.strip()


def show(title: str, text: str) -> None:
    print(f"\n{'=' * 70}\n{title}\n{'-' * 70}\n{text}")


# ============ setup: INDEX = load data/ -> cut into chunks -> turn every chunk into a vector ============
DATA_DIR = Path("data")   # written by the data/ cells at the top of this notebook


def chunk_text(text: str, chunk_size: int = 120, overlap: int = 20) -> list[str]:
    """Split text into chunks of ~chunk_size words; neighbouring chunks share `overlap` words."""
    words = text.split()
    if len(words) <= chunk_size:
        return [text.strip()]
    return [" ".join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size - overlap)]


def load_chunks(chunk_size: int = 120, overlap: int = 20) -> list[dict]:
    """Read every .md / .txt / .pdf file in data/ and chunk it. Each chunk remembers its file."""
    chunks = []
    for path in sorted(DATA_DIR.iterdir()):
        if path.suffix.lower() in {".md", ".txt"}:
            text = path.read_text(encoding="utf-8")
        elif path.suffix.lower() == ".pdf":
            from pypdf import PdfReader   # only needed when you put PDF files in data/
            text = "\n".join(page.extract_text() or "" for page in PdfReader(path).pages)
        else:
            continue
        for i, piece in enumerate(chunk_text(text, chunk_size, overlap)):
            chunks.append({"id": f"{path.name}#{i}", "source": path.name, "text": piece})
    return chunks


embedder = TextEmbedding("BAAI/bge-small-en-v1.5")   # free, runs on your CPU (small download the first time)


def embed(texts: list[str]) -> np.ndarray:
    """Texts -> a (n, dim) array of vectors, each scaled to length 1."""
    vectors = np.array(list(embedder.embed(texts)))
    return vectors / np.linalg.norm(vectors, axis=1, keepdims=True)


chunks = load_chunks()
vectors = embed([c["text"] for c in chunks])          # our whole "vector database": one numpy array
print(f"[index] {len(chunks)} chunks from {len({c['source'] for c in chunks})} files "
      f"| embeddings: local | llm: {PROVIDER}")


def search(query: str, k: int = 4) -> list[dict]:
    """Return the k most similar chunks, each with a 'score' (cosine similarity)."""
    scores = vectors @ embed([query])[0]              # dot product of length-1 vectors = cosine similarity
    return [{**chunks[i], "score": float(scores[i])} for i in np.argsort(-scores)[:k]]


ANSWER_SYSTEM = ("You answer questions using ONLY the provided context. "
                 "Cite sources like [1], [2]. If the answer is not in the context, say \"I don't know\".")


def answer_from_context(question: str, hits: list[dict]) -> str:
    context = "\n\n".join(f"[{i + 1}] (source: {h['source']})\n{h['text']}" for i, h in enumerate(hits))
    return chat(f"Context:\n{context}\n\nQuestion: {question}", system=ANSWER_SYSTEM)


# ============ the technique ============
QUESTION = "can I work from home?"


# --- technique A: multi-query ---
def multi_query_search(question: str, k: int = 4) -> list[dict]:
    rewrites = chat(
        f"Write 3 different search queries that would help answer: '{question}'. "
        "One per line, no numbering, no extra text."
    ).splitlines()
    queries = [question] + [q.strip("-• ").strip() for q in rewrites if q.strip()][:3]
    show("multi-query rewrites", "\n".join(queries))
    seen, merged = set(), []
    for q in queries:
        for hit in search(q, k=2):
            if hit["id"] not in seen:
                seen.add(hit["id"])
                merged.append(hit)
    return merged[:k]


# --- technique B: HyDE (Hypothetical Document Embeddings) ---
def hyde_search(question: str, k: int = 3) -> list[dict]:
    fake_answer = chat(f"Write a short paragraph from a company HR handbook that answers: {question}")
    show("HyDE hypothetical document", fake_answer)
    return search(fake_answer, k=k)


hits = multi_query_search(QUESTION)
show("answer (multi-query)", answer_from_context(QUESTION, hits))

hits = hyde_search(QUESTION)
show("answer (HyDE)", answer_from_context(QUESTION, hits))

## 05. `05_conversational_rag.py`

05 — conversational RAG  (RAG with memory)
follow-up questions like "and how much does IT cost?" make no sense on their own.
before searching, we ask the LLM to rewrite the follow-up into a STANDALONE question
using the chat history. then we do normal RAG, and keep the history for the next turn.

In [ ]:
import os
from pathlib import Path

import numpy as np
from dotenv import find_dotenv, load_dotenv
from fastembed import TextEmbedding
from openai import OpenAI

load_dotenv(find_dotenv(usecwd=True))   # read LLM_PROVIDER and your API key from the .env file

# ============ setup: the LLM (every provider speaks the OpenAI protocol, so one client works for all) ============
PROVIDERS = {  # name: (base_url, api-key env var, default model)
    "openai":     ("https://api.openai.com/v1", "OPENAI_API_KEY", "gpt-4o-mini"),
    "deepseek":   ("https://api.deepseek.com", "DEEPSEEK_API_KEY", "deepseek-chat"),
    "groq":       ("https://api.groq.com/openai/v1", "GROQ_API_KEY", "llama-3.3-70b-versatile"),
    "gemini":     ("https://generativelanguage.googleapis.com/v1beta/openai/", "GOOGLE_API_KEY", "gemini-2.5-flash"),
    "openrouter": ("https://openrouter.ai/api/v1", "OPENROUTER_API_KEY", "meta-llama/llama-3.3-70b-instruct:free"),
    "ollama":     (os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1"), None, "qwen2.5:7b"),
}
PROVIDER = os.getenv("LLM_PROVIDER", "groq").strip().lower()
BASE_URL, KEY_ENV, DEFAULT_MODEL = PROVIDERS[PROVIDER]
if KEY_ENV and not os.getenv(KEY_ENV):
    raise SystemExit(f"{KEY_ENV} is missing. Add it to your .env file (see .env.example).")
client = OpenAI(base_url=BASE_URL, api_key=os.getenv(KEY_ENV) if KEY_ENV else "ollama")
MODEL = os.getenv("LLM_MODEL") or DEFAULT_MODEL


def chat(prompt: str, system: str = "You are a helpful assistant.") -> str:
    """Send one prompt to the LLM and return its text answer."""
    resp = client.chat.completions.create(model=MODEL, temperature=0, messages=[
        {"role": "system", "content": system}, {"role": "user", "content": prompt}])
    return resp.choices[0].message.content.strip()


def show(title: str, text: str) -> None:
    print(f"\n{'=' * 70}\n{title}\n{'-' * 70}\n{text}")


# ============ setup: INDEX = load data/ -> cut into chunks -> turn every chunk into a vector ============
DATA_DIR = Path("data")   # written by the data/ cells at the top of this notebook


def chunk_text(text: str, chunk_size: int = 120, overlap: int = 20) -> list[str]:
    """Split text into chunks of ~chunk_size words; neighbouring chunks share `overlap` words."""
    words = text.split()
    if len(words) <= chunk_size:
        return [text.strip()]
    return [" ".join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size - overlap)]


def load_chunks(chunk_size: int = 120, overlap: int = 20) -> list[dict]:
    """Read every .md / .txt / .pdf file in data/ and chunk it. Each chunk remembers its file."""
    chunks = []
    for path in sorted(DATA_DIR.iterdir()):
        if path.suffix.lower() in {".md", ".txt"}:
            text = path.read_text(encoding="utf-8")
        elif path.suffix.lower() == ".pdf":
            from pypdf import PdfReader   # only needed when you put PDF files in data/
            text = "\n".join(page.extract_text() or "" for page in PdfReader(path).pages)
        else:
            continue
        for i, piece in enumerate(chunk_text(text, chunk_size, overlap)):
            chunks.append({"id": f"{path.name}#{i}", "source": path.name, "text": piece})
    return chunks


embedder = TextEmbedding("BAAI/bge-small-en-v1.5")   # free, runs on your CPU (small download the first time)


def embed(texts: list[str]) -> np.ndarray:
    """Texts -> a (n, dim) array of vectors, each scaled to length 1."""
    vectors = np.array(list(embedder.embed(texts)))
    return vectors / np.linalg.norm(vectors, axis=1, keepdims=True)


chunks = load_chunks()
vectors = embed([c["text"] for c in chunks])          # our whole "vector database": one numpy array
print(f"[index] {len(chunks)} chunks from {len({c['source'] for c in chunks})} files "
      f"| embeddings: local | llm: {PROVIDER}")


def search(query: str, k: int = 4) -> list[dict]:
    """Return the k most similar chunks, each with a 'score' (cosine similarity)."""
    scores = vectors @ embed([query])[0]              # dot product of length-1 vectors = cosine similarity
    return [{**chunks[i], "score": float(scores[i])} for i in np.argsort(-scores)[:k]]


ANSWER_SYSTEM = ("You answer questions using ONLY the provided context. "
                 "Cite sources like [1], [2]. If the answer is not in the context, say \"I don't know\".")


def answer_from_context(question: str, hits: list[dict]) -> str:
    context = "\n\n".join(f"[{i + 1}] (source: {h['source']})\n{h['text']}" for i, h in enumerate(hits))
    return chat(f"Context:\n{context}\n\nQuestion: {question}", system=ANSWER_SYSTEM)


# ============ the technique: memory + rewrite the follow-up into a standalone question ============
history: list[dict] = []  # the conversation memory: a plain list of messages


def ask(question: str) -> str:
    standalone = question
    if history:
        transcript = "\n".join(f"{m['role']}: {m['content']}" for m in history[-6:])
        standalone = chat(
            f"Chat history:\n{transcript}\n\nFollow-up question: {question}\n\n"
            "Rewrite the follow-up as a standalone question. Reply with the question only."
        )
    hits = search(standalone, k=3)
    answer = answer_from_context(standalone, hits)
    history.extend([{"role": "user", "content": question}, {"role": "assistant", "content": answer}])
    show(f"user: {question}\n(standalone: {standalone})", answer)
    return answer


ask("What kit do you sell for farms?")
ask("How much does it cost?")                  # "it" = SunBox Farm
ask("And what does the maintenance plan for it cost per year?")

## 06. `06_corrective_rag.py`

06 — Corrective RAG (CRAG)
naive RAG trusts whatever it retrieves. CRAG adds a GRADER step:
  1. retrieve chunks
  2. the LLM grades each chunk: is it relevant to the question? (yes/no)
  3. keep relevant chunks. if NONE are relevant -> "correct" by searching the web instead
  4. generate the answer from the corrected context

In [ ]:
import json
import os
import re
from pathlib import Path

import numpy as np
from ddgs import DDGS
from dotenv import find_dotenv, load_dotenv
from fastembed import TextEmbedding
from openai import OpenAI

load_dotenv(find_dotenv(usecwd=True))   # read LLM_PROVIDER and your API key from the .env file

# ============ setup: the LLM (every provider speaks the OpenAI protocol, so one client works for all) ============
PROVIDERS = {  # name: (base_url, api-key env var, default model)
    "openai":     ("https://api.openai.com/v1", "OPENAI_API_KEY", "gpt-4o-mini"),
    "deepseek":   ("https://api.deepseek.com", "DEEPSEEK_API_KEY", "deepseek-chat"),
    "groq":       ("https://api.groq.com/openai/v1", "GROQ_API_KEY", "llama-3.3-70b-versatile"),
    "gemini":     ("https://generativelanguage.googleapis.com/v1beta/openai/", "GOOGLE_API_KEY", "gemini-2.5-flash"),
    "openrouter": ("https://openrouter.ai/api/v1", "OPENROUTER_API_KEY", "meta-llama/llama-3.3-70b-instruct:free"),
    "ollama":     (os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1"), None, "qwen2.5:7b"),
}
PROVIDER = os.getenv("LLM_PROVIDER", "groq").strip().lower()
BASE_URL, KEY_ENV, DEFAULT_MODEL = PROVIDERS[PROVIDER]
if KEY_ENV and not os.getenv(KEY_ENV):
    raise SystemExit(f"{KEY_ENV} is missing. Add it to your .env file (see .env.example).")
client = OpenAI(base_url=BASE_URL, api_key=os.getenv(KEY_ENV) if KEY_ENV else "ollama")
MODEL = os.getenv("LLM_MODEL") or DEFAULT_MODEL


def chat(prompt: str, system: str = "You are a helpful assistant.") -> str:
    """Send one prompt to the LLM and return its text answer."""
    resp = client.chat.completions.create(model=MODEL, temperature=0, messages=[
        {"role": "system", "content": system}, {"role": "user", "content": prompt}])
    return resp.choices[0].message.content.strip()


def chat_json(prompt: str) -> dict:
    """Ask the LLM for JSON and parse it (tolerates ```json fences and extra words around it)."""
    text = chat(prompt, system="Reply with valid JSON only.")
    match = re.search(r"\{.*\}", text, re.DOTALL)
    try:
        return json.loads(match.group(0) if match else text)
    except json.JSONDecodeError:
        return {}


def web_search(query: str, max_results: int = 3) -> str:
    """Free web search via DuckDuckGo (no key). Returns plain text snippets."""
    try:
        results = DDGS().text(query, max_results=max_results)
        return "\n\n".join(f"{r['title']}: {r['body']} ({r['href']})" for r in results) or "no results"
    except Exception as e:  # free search is rate-limited; never crash the demo because of it
        return f"web search failed: {e}"


def show(title: str, text: str) -> None:
    print(f"\n{'=' * 70}\n{title}\n{'-' * 70}\n{text}")


# ============ setup: INDEX = load data/ -> cut into chunks -> turn every chunk into a vector ============
DATA_DIR = Path("data")   # written by the data/ cells at the top of this notebook


def chunk_text(text: str, chunk_size: int = 120, overlap: int = 20) -> list[str]:
    """Split text into chunks of ~chunk_size words; neighbouring chunks share `overlap` words."""
    words = text.split()
    if len(words) <= chunk_size:
        return [text.strip()]
    return [" ".join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size - overlap)]


def load_chunks(chunk_size: int = 120, overlap: int = 20) -> list[dict]:
    """Read every .md / .txt / .pdf file in data/ and chunk it. Each chunk remembers its file."""
    chunks = []
    for path in sorted(DATA_DIR.iterdir()):
        if path.suffix.lower() in {".md", ".txt"}:
            text = path.read_text(encoding="utf-8")
        elif path.suffix.lower() == ".pdf":
            from pypdf import PdfReader   # only needed when you put PDF files in data/
            text = "\n".join(page.extract_text() or "" for page in PdfReader(path).pages)
        else:
            continue
        for i, piece in enumerate(chunk_text(text, chunk_size, overlap)):
            chunks.append({"id": f"{path.name}#{i}", "source": path.name, "text": piece})
    return chunks


embedder = TextEmbedding("BAAI/bge-small-en-v1.5")   # free, runs on your CPU (small download the first time)


def embed(texts: list[str]) -> np.ndarray:
    """Texts -> a (n, dim) array of vectors, each scaled to length 1."""
    vectors = np.array(list(embedder.embed(texts)))
    return vectors / np.linalg.norm(vectors, axis=1, keepdims=True)


chunks = load_chunks()
vectors = embed([c["text"] for c in chunks])          # our whole "vector database": one numpy array
print(f"[index] {len(chunks)} chunks from {len({c['source'] for c in chunks})} files "
      f"| embeddings: local | llm: {PROVIDER}")


def search(query: str, k: int = 4) -> list[dict]:
    """Return the k most similar chunks, each with a 'score' (cosine similarity)."""
    scores = vectors @ embed([query])[0]              # dot product of length-1 vectors = cosine similarity
    return [{**chunks[i], "score": float(scores[i])} for i in np.argsort(-scores)[:k]]


ANSWER_SYSTEM = ("You answer questions using ONLY the provided context. "
                 "Cite sources like [1], [2]. If the answer is not in the context, say \"I don't know\".")


def answer_from_context(question: str, hits: list[dict]) -> str:
    context = "\n\n".join(f"[{i + 1}] (source: {h['source']})\n{h['text']}" for i, h in enumerate(hits))
    return chat(f"Context:\n{context}\n\nQuestion: {question}", system=ANSWER_SYSTEM)


# ============ the technique: grade every chunk, fall back to the web if none is relevant ============
def grade(question: str, chunk: dict) -> bool:
    verdict = chat_json(
        f"Question: {question}\n\nDocument:\n{chunk['text']}\n\n"
        'Does the document contain information useful for answering the question? '
        'Reply as JSON: {"relevant": "yes"} or {"relevant": "no"}'
    )
    return str(verdict.get("relevant", "no")).lower().startswith("y")


def corrective_rag(question: str) -> str:
    hits = search(question, k=4)
    relevant = [h for h in hits if grade(question, h)]
    show(f"grader kept {len(relevant)}/{len(hits)} chunks", "\n".join(h["id"] for h in relevant) or "(none)")
    if not relevant:
        print("-> no relevant internal documents, falling back to WEB SEARCH")
        relevant = [{"source": "web", "text": web_search(question)}]
    return answer_from_context(question, relevant)


# answerable from our documents
show("answer", corrective_rag("What does the DuneGuard coating do?"))
# NOT in our documents -> triggers the web fallback
show("answer", corrective_rag("What is the capital city of Libya and its approximate population?"))

## 07. `07_self_rag.py`

07 — Self-RAG  (the model checks its own answer)
after generating, we ask the LLM two questions about its own output:
  a) is the answer GROUNDED in the retrieved context? (no hallucination)
  b) does it actually ANSWER the question?
if either check fails, we retrieve more context and try again (max 3 attempts).

In [ ]:
import json
import os
import re
from pathlib import Path

import numpy as np
from dotenv import find_dotenv, load_dotenv
from fastembed import TextEmbedding
from openai import OpenAI

load_dotenv(find_dotenv(usecwd=True))   # read LLM_PROVIDER and your API key from the .env file

# ============ setup: the LLM (every provider speaks the OpenAI protocol, so one client works for all) ============
PROVIDERS = {  # name: (base_url, api-key env var, default model)
    "openai":     ("https://api.openai.com/v1", "OPENAI_API_KEY", "gpt-4o-mini"),
    "deepseek":   ("https://api.deepseek.com", "DEEPSEEK_API_KEY", "deepseek-chat"),
    "groq":       ("https://api.groq.com/openai/v1", "GROQ_API_KEY", "llama-3.3-70b-versatile"),
    "gemini":     ("https://generativelanguage.googleapis.com/v1beta/openai/", "GOOGLE_API_KEY", "gemini-2.5-flash"),
    "openrouter": ("https://openrouter.ai/api/v1", "OPENROUTER_API_KEY", "meta-llama/llama-3.3-70b-instruct:free"),
    "ollama":     (os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1"), None, "qwen2.5:7b"),
}
PROVIDER = os.getenv("LLM_PROVIDER", "groq").strip().lower()
BASE_URL, KEY_ENV, DEFAULT_MODEL = PROVIDERS[PROVIDER]
if KEY_ENV and not os.getenv(KEY_ENV):
    raise SystemExit(f"{KEY_ENV} is missing. Add it to your .env file (see .env.example).")
client = OpenAI(base_url=BASE_URL, api_key=os.getenv(KEY_ENV) if KEY_ENV else "ollama")
MODEL = os.getenv("LLM_MODEL") or DEFAULT_MODEL


def chat(prompt: str, system: str = "You are a helpful assistant.") -> str:
    """Send one prompt to the LLM and return its text answer."""
    resp = client.chat.completions.create(model=MODEL, temperature=0, messages=[
        {"role": "system", "content": system}, {"role": "user", "content": prompt}])
    return resp.choices[0].message.content.strip()


def chat_json(prompt: str) -> dict:
    """Ask the LLM for JSON and parse it (tolerates ```json fences and extra words around it)."""
    text = chat(prompt, system="Reply with valid JSON only.")
    match = re.search(r"\{.*\}", text, re.DOTALL)
    try:
        return json.loads(match.group(0) if match else text)
    except json.JSONDecodeError:
        return {}


def show(title: str, text: str) -> None:
    print(f"\n{'=' * 70}\n{title}\n{'-' * 70}\n{text}")


# ============ setup: INDEX = load data/ -> cut into chunks -> turn every chunk into a vector ============
DATA_DIR = Path("data")   # written by the data/ cells at the top of this notebook


def chunk_text(text: str, chunk_size: int = 120, overlap: int = 20) -> list[str]:
    """Split text into chunks of ~chunk_size words; neighbouring chunks share `overlap` words."""
    words = text.split()
    if len(words) <= chunk_size:
        return [text.strip()]
    return [" ".join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size - overlap)]


def load_chunks(chunk_size: int = 120, overlap: int = 20) -> list[dict]:
    """Read every .md / .txt / .pdf file in data/ and chunk it. Each chunk remembers its file."""
    chunks = []
    for path in sorted(DATA_DIR.iterdir()):
        if path.suffix.lower() in {".md", ".txt"}:
            text = path.read_text(encoding="utf-8")
        elif path.suffix.lower() == ".pdf":
            from pypdf import PdfReader   # only needed when you put PDF files in data/
            text = "\n".join(page.extract_text() or "" for page in PdfReader(path).pages)
        else:
            continue
        for i, piece in enumerate(chunk_text(text, chunk_size, overlap)):
            chunks.append({"id": f"{path.name}#{i}", "source": path.name, "text": piece})
    return chunks


embedder = TextEmbedding("BAAI/bge-small-en-v1.5")   # free, runs on your CPU (small download the first time)


def embed(texts: list[str]) -> np.ndarray:
    """Texts -> a (n, dim) array of vectors, each scaled to length 1."""
    vectors = np.array(list(embedder.embed(texts)))
    return vectors / np.linalg.norm(vectors, axis=1, keepdims=True)


chunks = load_chunks()
vectors = embed([c["text"] for c in chunks])          # our whole "vector database": one numpy array
print(f"[index] {len(chunks)} chunks from {len({c['source'] for c in chunks})} files "
      f"| embeddings: local | llm: {PROVIDER}")


def search(query: str, k: int = 4) -> list[dict]:
    """Return the k most similar chunks, each with a 'score' (cosine similarity)."""
    scores = vectors @ embed([query])[0]              # dot product of length-1 vectors = cosine similarity
    return [{**chunks[i], "score": float(scores[i])} for i in np.argsort(-scores)[:k]]


ANSWER_SYSTEM = ("You answer questions using ONLY the provided context. "
                 "Cite sources like [1], [2]. If the answer is not in the context, say \"I don't know\".")


def answer_from_context(question: str, hits: list[dict]) -> str:
    context = "\n\n".join(f"[{i + 1}] (source: {h['source']})\n{h['text']}" for i, h in enumerate(hits))
    return chat(f"Context:\n{context}\n\nQuestion: {question}", system=ANSWER_SYSTEM)


# ============ the technique: check the answer, widen the search and retry if it fails ============
def is_grounded(answer: str, hits: list[dict]) -> bool:
    context = "\n".join(h["text"] for h in hits)
    v = chat_json(f"Context:\n{context}\n\nAnswer:\n{answer}\n\n"
                  'Is every fact in the answer supported by the context? JSON: {"grounded": "yes" or "no"}')
    return str(v.get("grounded", "no")).lower().startswith("y")


def is_useful(question: str, answer: str) -> bool:
    v = chat_json(f"Question: {question}\nAnswer: {answer}\n\n"
                  'Does the answer resolve the question? JSON: {"useful": "yes" or "no"}')
    return str(v.get("useful", "no")).lower().startswith("y")


def self_rag(question: str, max_attempts: int = 3) -> str:
    k = 2
    for attempt in range(1, max_attempts + 1):
        hits = search(question, k=k)
        answer = answer_from_context(question, hits)
        grounded, useful = is_grounded(answer, hits), is_useful(question, answer)
        print(f"attempt {attempt}: k={k} grounded={grounded} useful={useful}")
        if grounded and useful:
            return answer
        k += 2  # not good enough -> widen the search and try again
    return answer + "\n(warning: could not fully verify this answer)"


show("answer", self_rag("Which project does Khaled Senussi lead, and how many farms were connected by end of 2025?"))

## 08. `08_graph_rag.py`

08 — Graph RAG  (a knowledge graph built by the LLM)
vector search finds SIMILAR text. but some questions need CONNECTIONS:
  "which projects are led by people who work in the lab that invented DuneGuard?"
graph RAG:
  1. the LLM reads every chunk and extracts (subject, relation, object) triples
  2. we store the triples in a graph (networkx)
  3. for a question, find the entities it mentions and walk their neighbours (2 hops)
  4. give those facts (+ normal vector hits) to the LLM
triples are cached in .cache/ so you only pay for extraction once.

In [ ]:
import json
import os
import re
from pathlib import Path

import networkx as nx
import numpy as np
from dotenv import find_dotenv, load_dotenv
from fastembed import TextEmbedding
from openai import OpenAI

load_dotenv(find_dotenv(usecwd=True))   # read LLM_PROVIDER and your API key from the .env file

# ============ setup: the LLM (every provider speaks the OpenAI protocol, so one client works for all) ============
PROVIDERS = {  # name: (base_url, api-key env var, default model)
    "openai":     ("https://api.openai.com/v1", "OPENAI_API_KEY", "gpt-4o-mini"),
    "deepseek":   ("https://api.deepseek.com", "DEEPSEEK_API_KEY", "deepseek-chat"),
    "groq":       ("https://api.groq.com/openai/v1", "GROQ_API_KEY", "llama-3.3-70b-versatile"),
    "gemini":     ("https://generativelanguage.googleapis.com/v1beta/openai/", "GOOGLE_API_KEY", "gemini-2.5-flash"),
    "openrouter": ("https://openrouter.ai/api/v1", "OPENROUTER_API_KEY", "meta-llama/llama-3.3-70b-instruct:free"),
    "ollama":     (os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1"), None, "qwen2.5:7b"),
}
PROVIDER = os.getenv("LLM_PROVIDER", "groq").strip().lower()
BASE_URL, KEY_ENV, DEFAULT_MODEL = PROVIDERS[PROVIDER]
if KEY_ENV and not os.getenv(KEY_ENV):
    raise SystemExit(f"{KEY_ENV} is missing. Add it to your .env file (see .env.example).")
client = OpenAI(base_url=BASE_URL, api_key=os.getenv(KEY_ENV) if KEY_ENV else "ollama")
MODEL = os.getenv("LLM_MODEL") or DEFAULT_MODEL


def chat(prompt: str, system: str = "You are a helpful assistant.") -> str:
    """Send one prompt to the LLM and return its text answer."""
    resp = client.chat.completions.create(model=MODEL, temperature=0, messages=[
        {"role": "system", "content": system}, {"role": "user", "content": prompt}])
    return resp.choices[0].message.content.strip()


def chat_json(prompt: str) -> dict:
    """Ask the LLM for JSON and parse it (tolerates ```json fences and extra words around it)."""
    text = chat(prompt, system="Reply with valid JSON only.")
    match = re.search(r"\{.*\}", text, re.DOTALL)
    try:
        return json.loads(match.group(0) if match else text)
    except json.JSONDecodeError:
        return {}


def show(title: str, text: str) -> None:
    print(f"\n{'=' * 70}\n{title}\n{'-' * 70}\n{text}")


# ============ setup: INDEX = load data/ -> cut into chunks -> turn every chunk into a vector ============
DATA_DIR = Path("data")   # written by the data/ cells at the top of this notebook


def chunk_text(text: str, chunk_size: int = 120, overlap: int = 20) -> list[str]:
    """Split text into chunks of ~chunk_size words; neighbouring chunks share `overlap` words."""
    words = text.split()
    if len(words) <= chunk_size:
        return [text.strip()]
    return [" ".join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size - overlap)]


def load_chunks(chunk_size: int = 120, overlap: int = 20) -> list[dict]:
    """Read every .md / .txt / .pdf file in data/ and chunk it. Each chunk remembers its file."""
    chunks = []
    for path in sorted(DATA_DIR.iterdir()):
        if path.suffix.lower() in {".md", ".txt"}:
            text = path.read_text(encoding="utf-8")
        elif path.suffix.lower() == ".pdf":
            from pypdf import PdfReader   # only needed when you put PDF files in data/
            text = "\n".join(page.extract_text() or "" for page in PdfReader(path).pages)
        else:
            continue
        for i, piece in enumerate(chunk_text(text, chunk_size, overlap)):
            chunks.append({"id": f"{path.name}#{i}", "source": path.name, "text": piece})
    return chunks


embedder = TextEmbedding("BAAI/bge-small-en-v1.5")   # free, runs on your CPU (small download the first time)


def embed(texts: list[str]) -> np.ndarray:
    """Texts -> a (n, dim) array of vectors, each scaled to length 1."""
    vectors = np.array(list(embedder.embed(texts)))
    return vectors / np.linalg.norm(vectors, axis=1, keepdims=True)


chunks = load_chunks()
vectors = embed([c["text"] for c in chunks])          # our whole "vector database": one numpy array
print(f"[index] {len(chunks)} chunks from {len({c['source'] for c in chunks})} files "
      f"| embeddings: local | llm: {PROVIDER}")


def search(query: str, k: int = 4) -> list[dict]:
    """Return the k most similar chunks, each with a 'score' (cosine similarity)."""
    scores = vectors @ embed([query])[0]              # dot product of length-1 vectors = cosine similarity
    return [{**chunks[i], "score": float(scores[i])} for i in np.argsort(-scores)[:k]]


ANSWER_SYSTEM = ("You answer questions using ONLY the provided context. "
                 "Cite sources like [1], [2]. If the answer is not in the context, say \"I don't know\".")


def answer_from_context(question: str, hits: list[dict]) -> str:
    context = "\n\n".join(f"[{i + 1}] (source: {h['source']})\n{h['text']}" for i, h in enumerate(hits))
    return chat(f"Context:\n{context}\n\nQuestion: {question}", system=ANSWER_SYSTEM)


# ============ the technique ============
CACHE = Path(".cache/graph_triples.json")


# --- 1. extract triples with the LLM ---
def extract_triples(text: str) -> list[list[str]]:
    data = chat_json(
        "Extract factual (subject, relation, object) triples about people, organisations, projects, "
        "places and products from the text. Use short names. "
        'JSON: {"triples": [["subject", "relation", "object"], ...]}\n\nText:\n' + text
    )
    return [t for t in data.get("triples", []) if isinstance(t, list) and len(t) == 3]


if CACHE.exists():
    triples = json.loads(CACHE.read_text())
else:
    triples = []
    for i, chunk in enumerate(chunks):
        print(f"extracting triples {i + 1}/{len(chunks)} ...")
        triples += extract_triples(chunk["text"])
    CACHE.parent.mkdir(exist_ok=True)
    CACHE.write_text(json.dumps(triples, indent=1))

# --- 2. build the graph ---
graph = nx.Graph()
for s, r, o in triples:
    graph.add_edge(str(s).strip(), str(o).strip(), relation=str(r).strip())
print(f"[graph] {graph.number_of_nodes()} entities, {graph.number_of_edges()} relations")


# --- 3. graph retrieval ---
def graph_facts(question: str, hops: int = 2, limit: int = 40) -> list[str]:
    names = chat(f"List the named entities (people, projects, places, products) in: '{question}'. "
                 "Comma-separated, nothing else.")
    wanted = [n.strip().lower() for n in names.split(",") if n.strip()]
    seeds = [node for node in graph.nodes if any(w in node.lower() or node.lower() in w for w in wanted)]
    nearby = set(seeds)
    for seed in seeds:
        nearby |= set(nx.single_source_shortest_path_length(graph, seed, cutoff=hops))
    facts = [f"{u} --{d['relation']}--> {v}" for u, v, d in graph.subgraph(nearby).edges(data=True)]
    return facts[:limit]


# --- 4. answer from graph facts + normal vector hits ---
QUESTION = "Who leads the lab where the DuneGuard coating was developed, and what project does that person lead?"

facts = graph_facts(QUESTION)
show("graph facts", "\n".join(facts) or "(no matching entities)")
context = [{"source": "knowledge-graph", "text": "\n".join(facts)}] + search(QUESTION, k=2)
show("answer (graph + vector)", answer_from_context(QUESTION, context))

## 09. `09_adaptive_rag.py`

09 — Adaptive RAG  (a router picks the strategy)
not every question needs the same pipeline:
  "hi, how are you?"                        -> no retrieval, just answer
  "what's our sick-leave policy?"           -> search our documents
  "who won the last football world cup?"    -> search the web
a ROUTER (one small LLM call) classifies the question first, then we run the right path.

In [ ]:
import json
import os
import re
from pathlib import Path

import numpy as np
from ddgs import DDGS
from dotenv import find_dotenv, load_dotenv
from fastembed import TextEmbedding
from openai import OpenAI

load_dotenv(find_dotenv(usecwd=True))   # read LLM_PROVIDER and your API key from the .env file

# ============ setup: the LLM (every provider speaks the OpenAI protocol, so one client works for all) ============
PROVIDERS = {  # name: (base_url, api-key env var, default model)
    "openai":     ("https://api.openai.com/v1", "OPENAI_API_KEY", "gpt-4o-mini"),
    "deepseek":   ("https://api.deepseek.com", "DEEPSEEK_API_KEY", "deepseek-chat"),
    "groq":       ("https://api.groq.com/openai/v1", "GROQ_API_KEY", "llama-3.3-70b-versatile"),
    "gemini":     ("https://generativelanguage.googleapis.com/v1beta/openai/", "GOOGLE_API_KEY", "gemini-2.5-flash"),
    "openrouter": ("https://openrouter.ai/api/v1", "OPENROUTER_API_KEY", "meta-llama/llama-3.3-70b-instruct:free"),
    "ollama":     (os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1"), None, "qwen2.5:7b"),
}
PROVIDER = os.getenv("LLM_PROVIDER", "groq").strip().lower()
BASE_URL, KEY_ENV, DEFAULT_MODEL = PROVIDERS[PROVIDER]
if KEY_ENV and not os.getenv(KEY_ENV):
    raise SystemExit(f"{KEY_ENV} is missing. Add it to your .env file (see .env.example).")
client = OpenAI(base_url=BASE_URL, api_key=os.getenv(KEY_ENV) if KEY_ENV else "ollama")
MODEL = os.getenv("LLM_MODEL") or DEFAULT_MODEL


def chat(prompt: str, system: str = "You are a helpful assistant.") -> str:
    """Send one prompt to the LLM and return its text answer."""
    resp = client.chat.completions.create(model=MODEL, temperature=0, messages=[
        {"role": "system", "content": system}, {"role": "user", "content": prompt}])
    return resp.choices[0].message.content.strip()


def chat_json(prompt: str) -> dict:
    """Ask the LLM for JSON and parse it (tolerates ```json fences and extra words around it)."""
    text = chat(prompt, system="Reply with valid JSON only.")
    match = re.search(r"\{.*\}", text, re.DOTALL)
    try:
        return json.loads(match.group(0) if match else text)
    except json.JSONDecodeError:
        return {}


def web_search(query: str, max_results: int = 3) -> str:
    """Free web search via DuckDuckGo (no key). Returns plain text snippets."""
    try:
        results = DDGS().text(query, max_results=max_results)
        return "\n\n".join(f"{r['title']}: {r['body']} ({r['href']})" for r in results) or "no results"
    except Exception as e:  # free search is rate-limited; never crash the demo because of it
        return f"web search failed: {e}"


def show(title: str, text: str) -> None:
    print(f"\n{'=' * 70}\n{title}\n{'-' * 70}\n{text}")


# ============ setup: INDEX = load data/ -> cut into chunks -> turn every chunk into a vector ============
DATA_DIR = Path("data")   # written by the data/ cells at the top of this notebook


def chunk_text(text: str, chunk_size: int = 120, overlap: int = 20) -> list[str]:
    """Split text into chunks of ~chunk_size words; neighbouring chunks share `overlap` words."""
    words = text.split()
    if len(words) <= chunk_size:
        return [text.strip()]
    return [" ".join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size - overlap)]


def load_chunks(chunk_size: int = 120, overlap: int = 20) -> list[dict]:
    """Read every .md / .txt / .pdf file in data/ and chunk it. Each chunk remembers its file."""
    chunks = []
    for path in sorted(DATA_DIR.iterdir()):
        if path.suffix.lower() in {".md", ".txt"}:
            text = path.read_text(encoding="utf-8")
        elif path.suffix.lower() == ".pdf":
            from pypdf import PdfReader   # only needed when you put PDF files in data/
            text = "\n".join(page.extract_text() or "" for page in PdfReader(path).pages)
        else:
            continue
        for i, piece in enumerate(chunk_text(text, chunk_size, overlap)):
            chunks.append({"id": f"{path.name}#{i}", "source": path.name, "text": piece})
    return chunks


embedder = TextEmbedding("BAAI/bge-small-en-v1.5")   # free, runs on your CPU (small download the first time)


def embed(texts: list[str]) -> np.ndarray:
    """Texts -> a (n, dim) array of vectors, each scaled to length 1."""
    vectors = np.array(list(embedder.embed(texts)))
    return vectors / np.linalg.norm(vectors, axis=1, keepdims=True)


chunks = load_chunks()
vectors = embed([c["text"] for c in chunks])          # our whole "vector database": one numpy array
print(f"[index] {len(chunks)} chunks from {len({c['source'] for c in chunks})} files "
      f"| embeddings: local | llm: {PROVIDER}")


def search(query: str, k: int = 4) -> list[dict]:
    """Return the k most similar chunks, each with a 'score' (cosine similarity)."""
    scores = vectors @ embed([query])[0]              # dot product of length-1 vectors = cosine similarity
    return [{**chunks[i], "score": float(scores[i])} for i in np.argsort(-scores)[:k]]


ANSWER_SYSTEM = ("You answer questions using ONLY the provided context. "
                 "Cite sources like [1], [2]. If the answer is not in the context, say \"I don't know\".")


def answer_from_context(question: str, hits: list[dict]) -> str:
    context = "\n\n".join(f"[{i + 1}] (source: {h['source']})\n{h['text']}" for i, h in enumerate(hits))
    return chat(f"Context:\n{context}\n\nQuestion: {question}", system=ANSWER_SYSTEM)


# ============ the technique: a router decides none / docs / web ============
ROUTER_PROMPT = """Route the user question to one of:
- "none":  greetings, chit-chat, general knowledge the model surely knows
- "docs":  anything about Sahara Sun Energy (products, prices, HR, projects, lab) or the SA-LIB workshop
- "web":   recent events or public facts not about the company
Question: {q}
JSON: {{"route": "none" | "docs" | "web"}}"""


def adaptive_rag(question: str) -> str:
    route = chat_json(ROUTER_PROMPT.format(q=question)).get("route", "docs")
    print(f"router -> {route}")
    if route == "none":
        return chat(question)
    if route == "web":
        return answer_from_context(question, [{"source": "web", "text": web_search(question)}])
    return answer_from_context(question, search(question, k=3))


for q in ["Hello! What can you help me with?",
          "How many days of sick leave do employees get?",
          "What is the latest stable version of Python?"]:
    show(q, adaptive_rag(q))

## 10. `10_agentic_rag.py`

10 — Agentic RAG  (no framework, just the tool-calling loop)
in every example so far, WE decided the steps. in agentic RAG the LLM decides:
  which tool to use (company docs? web? calculator?), what to search for,
  whether the results are enough, and when to stop.
this is the plan -> act -> observe loop from Day 2 — written by hand, ~40 lines.

In [ ]:
import json
import os
from pathlib import Path

import numpy as np
from ddgs import DDGS
from dotenv import find_dotenv, load_dotenv
from fastembed import TextEmbedding
from openai import OpenAI

load_dotenv(find_dotenv(usecwd=True))   # read LLM_PROVIDER and your API key from the .env file

# ============ setup: the LLM (every provider speaks the OpenAI protocol, so one client works for all) ============
PROVIDERS = {  # name: (base_url, api-key env var, default model)
    "openai":     ("https://api.openai.com/v1", "OPENAI_API_KEY", "gpt-4o-mini"),
    "deepseek":   ("https://api.deepseek.com", "DEEPSEEK_API_KEY", "deepseek-chat"),
    "groq":       ("https://api.groq.com/openai/v1", "GROQ_API_KEY", "llama-3.3-70b-versatile"),
    "gemini":     ("https://generativelanguage.googleapis.com/v1beta/openai/", "GOOGLE_API_KEY", "gemini-2.5-flash"),
    "openrouter": ("https://openrouter.ai/api/v1", "OPENROUTER_API_KEY", "meta-llama/llama-3.3-70b-instruct:free"),
    "ollama":     (os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1"), None, "qwen2.5:7b"),
}
PROVIDER = os.getenv("LLM_PROVIDER", "groq").strip().lower()
BASE_URL, KEY_ENV, DEFAULT_MODEL = PROVIDERS[PROVIDER]
if KEY_ENV and not os.getenv(KEY_ENV):
    raise SystemExit(f"{KEY_ENV} is missing. Add it to your .env file (see .env.example).")
client = OpenAI(base_url=BASE_URL, api_key=os.getenv(KEY_ENV) if KEY_ENV else "ollama")
MODEL = os.getenv("LLM_MODEL") or DEFAULT_MODEL


def show(title: str, text: str) -> None:
    print(f"\n{'=' * 70}\n{title}\n{'-' * 70}\n{text}")


# ============ setup: INDEX = load data/ -> cut into chunks -> turn every chunk into a vector ============
DATA_DIR = Path("data")   # written by the data/ cells at the top of this notebook


def chunk_text(text: str, chunk_size: int = 120, overlap: int = 20) -> list[str]:
    """Split text into chunks of ~chunk_size words; neighbouring chunks share `overlap` words."""
    words = text.split()
    if len(words) <= chunk_size:
        return [text.strip()]
    return [" ".join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size - overlap)]


def load_chunks(chunk_size: int = 120, overlap: int = 20) -> list[dict]:
    """Read every .md / .txt / .pdf file in data/ and chunk it. Each chunk remembers its file."""
    chunks = []
    for path in sorted(DATA_DIR.iterdir()):
        if path.suffix.lower() in {".md", ".txt"}:
            text = path.read_text(encoding="utf-8")
        elif path.suffix.lower() == ".pdf":
            from pypdf import PdfReader   # only needed when you put PDF files in data/
            text = "\n".join(page.extract_text() or "" for page in PdfReader(path).pages)
        else:
            continue
        for i, piece in enumerate(chunk_text(text, chunk_size, overlap)):
            chunks.append({"id": f"{path.name}#{i}", "source": path.name, "text": piece})
    return chunks


embedder = TextEmbedding("BAAI/bge-small-en-v1.5")   # free, runs on your CPU (small download the first time)


def embed(texts: list[str]) -> np.ndarray:
    """Texts -> a (n, dim) array of vectors, each scaled to length 1."""
    vectors = np.array(list(embedder.embed(texts)))
    return vectors / np.linalg.norm(vectors, axis=1, keepdims=True)


chunks = load_chunks()
vectors = embed([c["text"] for c in chunks])          # our whole "vector database": one numpy array
print(f"[index] {len(chunks)} chunks from {len({c['source'] for c in chunks})} files "
      f"| embeddings: local | llm: {PROVIDER}")


def search(query: str, k: int = 4) -> list[dict]:
    """Return the k most similar chunks, each with a 'score' (cosine similarity)."""
    scores = vectors @ embed([query])[0]              # dot product of length-1 vectors = cosine similarity
    return [{**chunks[i], "score": float(scores[i])} for i in np.argsort(-scores)[:k]]


# ============ the technique: tools + the plan -> act -> observe loop ============
# --- tools: plain python functions ---
def search_company_docs(query: str) -> str:
    hits = search(query, k=3)
    return "\n\n".join(f"[{i + 1}] (source: {h['source']})\n{h['text']}" for i, h in enumerate(hits))


def web_search(query: str, max_results: int = 3) -> str:
    """Free web search via DuckDuckGo (no key). Returns plain text snippets."""
    try:
        results = DDGS().text(query, max_results=max_results)
        return "\n\n".join(f"{r['title']}: {r['body']} ({r['href']})" for r in results) or "no results"
    except Exception as e:  # free search is rate-limited; never crash the demo because of it
        return f"web search failed: {e}"


def calculator(expression: str) -> str:
    return str(eval(expression, {"__builtins__": {}}, {}))  # demo only: arithmetic like "18500*0.2"


TOOLS = {"search_company_docs": search_company_docs, "web_search": web_search, "calculator": calculator}

# --- tool descriptions the LLM reads to decide WHEN to use each tool ---
TOOL_SPECS = [
    {"type": "function", "function": {
        "name": "search_company_docs",
        "description": "Search Sahara Sun Energy internal documents (products, prices, HR policy, projects, lab).",
        "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}}},
    {"type": "function", "function": {
        "name": "web_search",
        "description": "Search the public web for current or general information.",
        "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}}},
    {"type": "function", "function": {
        "name": "calculator",
        "description": "Evaluate an arithmetic expression, e.g. '34000*0.2'.",
        "parameters": {"type": "object", "properties": {"expression": {"type": "string"}}, "required": ["expression"]}}},
]

SYSTEM = ("You are a research assistant for Sahara Sun Energy. Use tools to find facts before answering. "
          "You may call tools several times. Cite sources. Be concise.")


def agent(question: str, max_steps: int = 6) -> str:
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    for step in range(max_steps):
        msg = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOL_SPECS).choices[0].message
        if not msg.tool_calls:                       # no tool requested -> this is the final answer
            return msg.content
        messages.append(msg)                          # PLAN: the model asked for tools
        for call in msg.tool_calls:                   # ACT: run each tool
            args = json.loads(call.function.arguments or "{}")
            print(f"step {step + 1}: {call.function.name}({args})")
            try:
                result = TOOLS[call.function.name](**args)
            except Exception as e:
                result = f"tool error: {e}"
            messages.append({"role": "tool", "tool_call_id": call.id, "content": str(result)})  # OBSERVE
    return "stopped: too many steps"


show("answer", agent(
    "A farmer wants the SunBox Farm kit with the Pay-as-you-save plan. "
    "How much is the deposit, and how much is each monthly instalment?"
))

## 11. `11_agentic_rag_langgraph.py`

11 — Agentic RAG with LangGraph
the same agent as 10_agentic_rag.py, but the loop is a LangGraph GRAPH:

    START -> [agent] --(wants a tool?)--> [tools] --+
                ^                                    |
                +------------------------------------+
             [agent] --(no tool call)--> END

LangGraph gives you: a visible graph, memory (checkpointer), streaming, human-in-the-loop.

In [ ]:
import os
from pathlib import Path

import numpy as np
from ddgs import DDGS
from dotenv import find_dotenv, load_dotenv
from fastembed import TextEmbedding
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

load_dotenv(find_dotenv(usecwd=True))   # read LLM_PROVIDER and your API key from the .env file

# ============ setup: the LLM (every provider speaks the OpenAI protocol, so ChatOpenAI works for all) ============
PROVIDERS = {  # name: (base_url, api-key env var, default model)
    "openai":     ("https://api.openai.com/v1", "OPENAI_API_KEY", "gpt-4o-mini"),
    "deepseek":   ("https://api.deepseek.com", "DEEPSEEK_API_KEY", "deepseek-chat"),
    "groq":       ("https://api.groq.com/openai/v1", "GROQ_API_KEY", "llama-3.3-70b-versatile"),
    "gemini":     ("https://generativelanguage.googleapis.com/v1beta/openai/", "GOOGLE_API_KEY", "gemini-2.5-flash"),
    "openrouter": ("https://openrouter.ai/api/v1", "OPENROUTER_API_KEY", "meta-llama/llama-3.3-70b-instruct:free"),
    "ollama":     (os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1"), None, "qwen2.5:7b"),
}
PROVIDER = os.getenv("LLM_PROVIDER", "groq").strip().lower()
BASE_URL, KEY_ENV, DEFAULT_MODEL = PROVIDERS[PROVIDER]
if KEY_ENV and not os.getenv(KEY_ENV):
    raise SystemExit(f"{KEY_ENV} is missing. Add it to your .env file (see .env.example).")
llm = ChatOpenAI(model=os.getenv("LLM_MODEL") or DEFAULT_MODEL, base_url=BASE_URL,
                 api_key=os.getenv(KEY_ENV) if KEY_ENV else "ollama", temperature=0)


def show(title: str, text: str) -> None:
    print(f"\n{'=' * 70}\n{title}\n{'-' * 70}\n{text}")


# ============ setup: INDEX = load data/ -> cut into chunks -> turn every chunk into a vector ============
DATA_DIR = Path("data")   # written by the data/ cells at the top of this notebook


def chunk_text(text: str, chunk_size: int = 120, overlap: int = 20) -> list[str]:
    """Split text into chunks of ~chunk_size words; neighbouring chunks share `overlap` words."""
    words = text.split()
    if len(words) <= chunk_size:
        return [text.strip()]
    return [" ".join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size - overlap)]


def load_chunks(chunk_size: int = 120, overlap: int = 20) -> list[dict]:
    """Read every .md / .txt / .pdf file in data/ and chunk it. Each chunk remembers its file."""
    chunks = []
    for path in sorted(DATA_DIR.iterdir()):
        if path.suffix.lower() in {".md", ".txt"}:
            text = path.read_text(encoding="utf-8")
        elif path.suffix.lower() == ".pdf":
            from pypdf import PdfReader   # only needed when you put PDF files in data/
            text = "\n".join(page.extract_text() or "" for page in PdfReader(path).pages)
        else:
            continue
        for i, piece in enumerate(chunk_text(text, chunk_size, overlap)):
            chunks.append({"id": f"{path.name}#{i}", "source": path.name, "text": piece})
    return chunks


embedder = TextEmbedding("BAAI/bge-small-en-v1.5")   # free, runs on your CPU (small download the first time)


def embed(texts: list[str]) -> np.ndarray:
    """Texts -> a (n, dim) array of vectors, each scaled to length 1."""
    vectors = np.array(list(embedder.embed(texts)))
    return vectors / np.linalg.norm(vectors, axis=1, keepdims=True)


chunks = load_chunks()
vectors = embed([c["text"] for c in chunks])          # our whole "vector database": one numpy array
print(f"[index] {len(chunks)} chunks from {len({c['source'] for c in chunks})} files "
      f"| embeddings: local | llm: {PROVIDER}")


def search(query: str, k: int = 4) -> list[dict]:
    """Return the k most similar chunks, each with a 'score' (cosine similarity)."""
    scores = vectors @ embed([query])[0]              # dot product of length-1 vectors = cosine similarity
    return [{**chunks[i], "score": float(scores[i])} for i in np.argsort(-scores)[:k]]


# ============ the technique: tools + a LangGraph loop ============
@tool
def search_company_docs(query: str) -> str:
    """Search Sahara Sun Energy internal documents (products, prices, HR policy, projects, lab)."""
    hits = search(query, k=3)
    return "\n\n".join(f"[{i + 1}] (source: {h['source']})\n{h['text']}" for i, h in enumerate(hits))


@tool
def web_search(query: str) -> str:
    """Search the public web for current or general information."""
    try:  # free DuckDuckGo search (no key); it is rate-limited, so never crash the demo because of it
        results = DDGS().text(query, max_results=3)
        return "\n\n".join(f"{r['title']}: {r['body']} ({r['href']})" for r in results) or "no results"
    except Exception as e:
        return f"web search failed: {e}"


tools = [search_company_docs, web_search]
llm_with_tools = llm.bind_tools(tools)


# node 1: the agent (the LLM decides: answer, or call a tool)
def agent_node(state: MessagesState):
    system = {"role": "system", "content": "Use tools to find facts before answering. Cite sources."}
    return {"messages": [llm_with_tools.invoke([system] + state["messages"])]}


# build the graph
graph = StateGraph(MessagesState)
graph.add_node("agent", agent_node)
graph.add_node("tools", ToolNode(tools))       # node 2: runs whichever tools the agent asked for
graph.add_edge(START, "agent")
graph.add_conditional_edges("agent", tools_condition)  # tool call? -> "tools", else -> END
graph.add_edge("tools", "agent")               # after tools, go back to the agent (the loop)
app = graph.compile(checkpointer=InMemorySaver())  # checkpointer = memory between turns

print(app.get_graph().draw_mermaid())  # paste into https://mermaid.live to see the graph

config = {"configurable": {"thread_id": "farmer-1"}}
for question in ["What does the Green Wells Program do and who leads it?",
                 "How many farms has it connected so far, out of how many planned?"]:  # follow-up uses memory
    result = app.invoke({"messages": [{"role": "user", "content": question}]}, config)
    show(question, result["messages"][-1].content)

---
*SA-LIB AI Workshop 2026 · Day 1*